In [1]:
import pandas as pd

数据透视
核心思想转换视角,把长表转换人类阅读的宽表.指定一列数据让它变成多列

pivot  重复会报错
- index 指定列位行索引
- columns 指定那列为列索引
- value 那一列位单元格数据

pivot_table 不仅改变数据形状,还可以自带聚合功能
- data
- index
- values
- columns
- aggfunc mean,sum
- fill_value
- margin 边缘汇总



In [3]:

# 长表数据
df = pd.DataFrame({
    '日期': ['周一', '周一', '周二', '周二','周三'],
    '城市': ['北京', '上海', '北京', '上海','上海'],
    '温度': [25, 28, 26, 29,40]
})
# 分维度
df.pivot(index='城市',columns='日期',values='温度')

日期,周一,周三,周二
城市,,,
上海,28.0,40.0,29.0
北京,25.0,NaN,26.0


In [ ]:
# 含有重复记录的长表
df_sales = pd.DataFrame({
    '日期': ['周一', '周一', '周一', '周二'],
    '商品': ['苹果', '苹果', '香蕉', '苹果'],
    '销量': [10, 5, 20, 15]
})

res_pivot_table = pd.pivot_table(
    df_sales, 
    index='日期', 
    columns='商品', 
    values='销量', 
    aggfunc='sum',     # 将重复的记录相加
    fill_value=0,      # 没有发生交易的格子填 0
    margins=True       # 开启总计
)
print(res_pivot_table)


# df_sales.pivot(index='日期',columns='商品',values='销量')

商品   苹果  香蕉  All
日期              
周一   15  20   35
周二   15   0   15
All  30  20   50


ValueError: Index contains duplicate entries, cannot reshape

数据堆叠
将列标签转换为行索引的外层,stack后会变成一个带有层次化的索引;将宽表转换存入关系数据库的长表

stack(列名->行索引) 数据库入库格式化
- levels 控制要堆叠那一层列
- dropna

unstack(行索引->列名) group聚和后生成的交叉报表


In [44]:

# 构造一个宽表（比如常见的年度报表）
df_wide = pd.DataFrame({
    '北京': [100, 120],
    '上海': [150, 180]
}, index=['2023', '2024'])

print("=== 原始宽表 ===")
print(df_wide)

print("\n=== 使用 stack() 堆叠后 ===")
res_stacked = df_wide.stack()
print(res_stacked)

=== 原始宽表 ===
       北京   上海
2023  100  150
2024  120  180

=== 使用 stack() 堆叠后 ===
2023  北京    100
      上海    150
2024  北京    120
      上海    180
dtype: int64


In [ ]:
res_stacked.reset_index() #把堆叠编程普通列

,level_0,level_1,0
0,2023,北京,100
1,2023,上海,150
2,2024,北京,120
3,2024,上海,180


In [46]:
res_stacked.unstack()

,北京,上海
2023,100,150
2024,120,180


In [ ]:
import plotly.express.data as ped
iris=ped.iris()
# 行索引变成列名 
un_irsi=iris.groupby('species').mean().unstack()
un_irsi['sepal_length'] 

species
setosa        5.006
versicolor    5.936
virginica     6.588
dtype: float64

In [65]:
pd.DataFrame(un_irsi.reset_index().set_index(['species','level_0']).unstack())

0                                                
level_0    petal_length petal_width sepal_length sepal_width species_id
species                                                                
setosa            1.464       0.244        5.006       3.418        1.0
versicolor        4.260       1.326        5.936       2.770        2.0
virginica         5.552       2.026        6.588       2.974        3.0

交叉表crosstab分类计算
计算两个分类变量的交叉频率表,与piovt_table很像,但是它默认行为是计数

- index 分组的行,可以是Series
- colunms 分组列
- values 如果不想计数,可以传入某个数值进行聚合操作,且必须配合aggfunc
- margins 会添加总计
- margins_name 汇总名称
- normalize 转换成百分比 bool/index按行求均;colunm按列;all按所有

In [ ]:
df_sales = pd.DataFrame({
    '日期': ['周一', '周一', '周一', '周二'],
    '商品': ['苹果', '苹果', '香蕉', '苹果'],
    '销量': [10, 5, 20, 15]
})

pd.crosstab(index=df_sales['日期'],columns=df_sales['商品'],normalize='index')

商品,苹果,香蕉
日期,,
周一,0.666667,0.333333
周二,1.000000,0.000000


数据转制
行转换列,.T或transpose.转换后可能会类型发生变化

swapaxes 废弃

In [ ]:
df_sales = pd.DataFrame({
    '日期': ['周一', '周一', '周一', '周二'],
    '商品': ['苹果', '香蕉', '茶叶', '苹果'],
    '销量': [10, 5, 20, 15]
})
df_sales.pivot(columns='日期',index='商品',values='销量').fillna(0)

日期,周一,周二
商品,,
苹果,10.0,15.0
茶叶,20.0,0.0
香蕉,5.0,0.0


In [80]:
df_sales.transpose()

,0,1,2,3
日期,周一,周一,周一,周二
商品,苹果,香蕉,茶叶,苹果
销量,10,5,20,15


In [ ]:
df_sales.swapaxes('index','columns')

/var/folders/f7/rx7cmvdx6nl86z_s7s86w4zm0000gn/T/ipykernel_4495/1669805521.py:1: FutureWarning:

'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.



,0,1,2,3
日期,周一,周一,周一,周二
商品,苹果,香蕉,茶叶,苹果
销量,10,5,20,15


数据融合
melt把宽表变成长表,与piovt逆操作.他们都依赖于列,而stack/unstack依赖于索引
- id_vars 保持不变的列,作为标识符保留的列
- value_vars 需要被融化的列,如果不指定末日id_vars以外的
- var_name 存放原列名的名字
- value_name 存放数值列的名字

In [86]:
df = pd.DataFrame({
    '姓名': ['张三', '李四'],
    '语文': [90, 85],
    '数学': [95, 88]
})

# 保持姓名不变，把语文、数学融化成"学科"和"分数"
res_melt = df.melt(id_vars=['姓名'], var_name='学科', value_name='分数')
print(res_melt)

   姓名  学科  分数
0  张三  语文  90
1  李四  语文  85
2  张三  数学  95
3  李四  数学  88


In [87]:
df

,姓名,语文,数学
0,张三,90,95
1,李四,85,88


In [92]:
df.melt(id_vars='姓名',value_vars=['数学'])

,姓名,variable,value
0,张三,数学,95
1,李四,数学,88


虚拟变量
在数据分析中,模型无法处理中文或英文字符(比如颜色里有红,黄,蓝).虚拟变量的作用把这类分类文本数据转换成0/1

- data
- columns 对那列进行转换
- prefix 生成新列前缀
- drop_first 

In [ ]:

df_color = pd.DataFrame({'物品': ['苹果', '香蕉', '蓝莓'], '颜色': ['红', '黄', '蓝']})

# 将颜色转换为虚拟变量
res_dummies = pd.get_dummies(df_color, columns=['颜色'], prefix='is')
print(res_dummies)

print('-'*10,'repeate')
food=pd.Categorical(['苹果', '香蕉', '蓝莓','苹果'],categories=['苹果', '香蕉', '蓝莓'])

df_color = pd.DataFrame({'物品': food,
                          '颜色': ['红', '黄', '蓝','黄']},index=food
                         )

res_dummies = pd.get_dummies(df_color, columns=['颜色'], prefix='is')
# 合并属性
res_dummies.groupby('物品',observed=True).max()

   物品   is_红   is_蓝   is_黄
0  苹果   True  False  False
1  香蕉  False  False   True
2  蓝莓  False   True  False
---------- repeate


,is_红,is_蓝,is_黄
物品,,,
苹果,True,False,True
香蕉,False,False,True
蓝莓,False,True,False


因子化 给类别贴标签
它直接把文本映射成标签(0,1,...),返回两个元素的元组,转换后的整数数组,唯一的类别列表

In [105]:
df_level = pd.DataFrame({'用户ID': [1, 2, 3, 4], '等级': ['青铜', '黄金', '白银', '青铜']})

# 进行因子化
labels, unique_values = pd.factorize(df_level['等级'])

# 将生成的整数标签赋值回原表
df_level['等级_ID'] = labels
print(df_level)
print(unique_values) # 输出 ['青铜', '黄金', '白银']

   用户ID  等级  等级_ID
0     1  青铜      0
1     2  黄金      1
2     3  白银      2
3     4  青铜      0
Index(['青铜', '黄金', '白银'], dtype='object')


爆炸列表:释放折叠的数据
把列表里的元素打开,每个个元素占单独行
- columns 要爆炸的列
- ignore_index 爆炸后会产生多个共享同一个旧索引

In [107]:



data=pd.DataFrame({'物品':[1,2,[3,4],4],'编号':['A','B','C','D']})

data.explode(column='物品')

,物品,编号
0,1,A
1,2,B
2,3,C
2,4,C
3,4,D
